# Benchmark Académico: Evaluación Comparativa de LLMs Locales en Quechua Collao
**Paper**: *Morpho-Semantic Scaffolding vs. Naive Parameter Scaling in Low-Resource Polysynthetic Languages (Quechua)*

Este notebook ejecuta la evaluación empírica de traducción Quechua -> Español para múltiples modelos abiertos servidos localmente en GPU mediante **Ollama**.

### ⚙️ Instrucciones Previas:
1. Ve a **Entorno de ejecución > Cambiar tipo de entorno de ejecución**.
2. Selecciona **GPU** (T4, L4 o A100 según tu suscripción Colab Pro/Plus).
3. Ejecuta las celdas en orden secuencial.

In [1]:
# 1. Verificar aceleración GPU disponible
!nvidia-smi

Fri Oct 16 17:58:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             48W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
# 2. Clonar el repositorio e instalar dependencias
import os
repo_url = "https://github.com/RodrigoAlexander7/generative-quechua-translator.git"
if not os.path.exists("generative-quechua-translator"):
    !git clone {repo_url}
%cd generative-quechua-translator
!pip install -q -r requirements.txt
!pip install -q tabulate sacrebleu pandas

Cloning into 'generative-quechua-translator'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 140 (delta 43), reused 132 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 5.76 MiB | 27.95 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/generative-quechua-translator/generative-quechua-translator/generative-quechua-translator


In [9]:
# 3. Instalar y levantar Ollama con soporte CUDA en segundo plano
import subprocess, time, urllib.request

# Ollama requiere zstd para descomprimir su instalador
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el daemon de Ollama (log en /content/ollama.log)
log = open("/content/ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=log)

# Esperar hasta que el servidor responda (máx. 60 s)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print("[+] Ollama está listo")
        break
    except Exception:
        time.sleep(1)
else:
    print("[!] Ollama no respondió; revisa /content/ollama.log")

!ollama list

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
[+] Ollama está listo
NAME           ID              SIZE      MODIFIED           
llama3.2:3b    a80c4f17acd5    2.0 GB    About a minute ago    


In [10]:
# 4. Descargar los modelos para el Paper
# Puedes comentar o descomentar según la VRAM disponible (T4: 16GB, A100: 40GB)
models_to_pull = [
    "llama3.2:3b",       # 3.2B (Edge)
    "llama3.1:8b",       # 8.0B (Baseline de facto)
    "qwen2.5:7b",        # 7.6B (Multilingüe denso)
    "qwen2.5:14b",       # 14.7B (Sweet spot para A100/L4)
    "aya-expanse:8b",    # 8.0B (Especialista en lenguas originarias)
    "deepseek-r1:8b",  # 8.0B (Paradigma de razonamiento CoT)
    "qwen2.5:32b",     # 32B (Solo en A100 con 40GB VRAM)
]

for m in models_to_pull:
    print(f"[*] Descargando modelo: {m}...")
    !ollama pull {m}

!ollama list

[*] Descargando modelo: llama3.2:3b...

[*] Descargando modelo: llama3.1:8b...

[*] Descargando modelo: qwen2.5:7b...

[*] Descargando modelo: qwen2.5:14b...

[*] Descargando modelo: aya-expanse:8b...

[*] Descargando modelo: deepseek-r1:8b...

[*] Descargando modelo: qwen2.5:32b...

NAME              ID              SIZE      MODIFIED               
qwen2.5:32b       9f13ba1299af    19 GB     Less than a second ago    
deepseek-r1:8b    6995872bfe4c    5.2 GB    2 minutes ago             
aya-expanse:8b    65f986688a01    5.1 GB    2 minutes ago             
qwen2.5:14b       7cdf5a0187d5    9.0 GB    3 minutes ago             
qwen2.5:7b        845dbda0ea48    4.7 GB    4 minutes ago             
llama3.1:8b       46e0c10c039e    4.9 GB    5 minutes ago             
llama3.2:3b       a80c4f17acd5    2.0 GB    5 minutes ago             


In [11]:
# 5. Prueba de humo: Traducir una oración compleja con andamiaje compacto (-L)
test_sentence = "Allin yuyaymanaywan rimanakuspallanchisraqmi, paqtataq kay sasachakuyninchiskunata allichasunchis churanakuymanta ayqenanchispaq."
!python3 bin/qtr translate "{test_sentence}" --provider ollama --model qwen2.5:7b -L


[*] Procesando traducción generativa...
Oración Quechua: "Allin yuyaymanaywan rimanakuspallanchisraqmi, paqtataq kay sasachakuyninchiskunata allichasunchis churanakuymanta ayqenanchispaq."

┌─ TRADUCCIÓN AL ESPAÑOL [Motor: OLLAMA (qwen2.5:7b)]
│  Esta conversación entre varios buenos yuyaymanaywan, que iguala y sasachakuyninchiskunata allichasunchis churanakuymanta ayqenanchispaq.
└──────────────────────────────────────────────────────────

Análisis Morfo-Semántico:
  • Glosa Interlineal:   Allin yuyaymanaywan (Bueno yuyaymanaywan) rimanakuspallanchisraqmi (conversación entre varias personas directo) paqtataq (iguala) kay (y) sasachakuyninchiskunata (ponerse algo difícil de realizar o lograr) allichasunchis (arreglar) churanakuymanta (desafiarse) ayqenanchispaq (persona u objeto cuya presencia provoca huida o evasión).
  • Evidencialidad:      Evidencialidad directa (-mi)
  • Clusividad:          Inclusivo (hablante + oyente) -nchis
  • Justificación:       La oración se refiere a una

In [12]:
# 6. Actualizar código con métricas de paper y ejecutar las 100 oraciones completas
!git pull origin main

# Lista de modelos descargados en tu Celda 4
eval_models = "llama3.2:3b,llama3.1:8b,qwen2.5:7b,qwen2.5:14b,aya-expanse:8b"

# --start 0 --end 100 evalúa TODO el dataset (100 oraciones x 2 modos x 5 modelos = 1000 inferencias)
!python3 benchmark/multi_model_runner.py \
    --models "{eval_models}" \
    --start 0 \
    --end 100 \
    --output-dir data/benchmark_results

remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 11 (delta 6), reused 11 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 9.50 KiB | 2.38 MiB/s, done.
From https://github.com/RodrigoAlexander7/generative-quechua-translator
 * branch            main       -> FETCH_HEAD
   3e7066b..18ece69  main       -> origin/main
Updating 3e7066b..18ece69
Fast-forward
 benchmark/academic_metrics.py               | 198 ++++---
 benchmark/dataset.json                      | 802 ++++++++++++++++++++++++----
 benchmark/multi_model_runner.py             | 101 ++--
 notebooks/colab_multi_model_benchmark.ipynb |  14 +-
 src/translation/prompt_templates.py         |   1 +
 5 files changed, 879 insertions(+), 237 deletions(-)

[*] INICIANDO BENCHMARK MULTI-MODELO (ACADÉMICO)
[*] Oraciones a evaluar: 100 (índices 0 a 99)
[*] Modelos seleccionados: ['llama3.2:3b', 'llama3.1:8b', 'qwen2.5:7b', 'qwen2

In [13]:
# Actualizar el runner con soporte de fusión incremental (no borra los 5 modelos ya corridos)
!git pull origin main

# 1. Descargar el modelo grande (32B) y/o el modelo de razonamiento que faltaban
missing_models = [
    "qwen2.5:32b",      # Frontera abierta de 32B (entra sobrado en tus 80GB de VRAM)
    "deepseek-r1:8b",   # Modelo de razonamiento (CoT)
    # "qwen3.5:9b",     # Descomenta si también quieres incluir Qwen 3.5 9B
]

for m in missing_models:
    print(f"[*] Descargando {m}...")
    !ollama pull {m}

# 2. Correr SOLO los modelos faltantes (se fusionarán automáticamente con los 5 ya evaluados)
models_str = ",".join(missing_models)
!python3 benchmark/multi_model_runner.py \
    --models "{models_str}" \
    --start 0 \
    --end 100 \
    --output-dir data/benchmark_results

remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 12 (delta 7), reused 11 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 10.00 KiB | 2.50 MiB/s, done.
From https://github.com/RodrigoAlexander7/generative-quechua-translator
 * branch            main       -> FETCH_HEAD
   18ece69..167a88c  main       -> origin/main
Updating 18ece69..167a88c
Fast-forward
 benchmark/multi_model_runner.py             |  89 +++++---
 notebooks/colab_multi_model_benchmark.ipynb |  41 +++-
 src/segmentation/peeler.py                  |  15 +-
 src/segmentation/scaffold.py                | 301 ++++++++++++++++++++++++++--
 4 files changed, 390 insertions(+), 56 deletions(-)
[*] Descargando qwen2.5:32b...

[*] Descargando deepseek-r1:8b...


[*] INICIANDO BENCHMARK MULTI-MODELO (ACADÉMICO)
[*] Oraciones a evaluar: 100 (índices 0 a 99)
[*] Modelos seleccionados: ['qwen2.5:32b', 'deepseek-r1:8b']

[

In [14]:
# 7. Visualizar Resumen de Métricas del Paper y Tabla Comparativa Lado a Lado
import json
import pandas as pd
from IPython.display import display, HTML

# Evitar que Colab corte el texto de las traducciones o esconda columnas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

# A) Tabla 1: Resumen Agregado y Significancia Estadística (Bootstrap)
with open("data/benchmark_results/multi_model_benchmark.json", "r", encoding="utf-8") as f:
    bench_data = json.load(f)

summary_rows = []
for model_name, stats in bench_data["summary"].items():
    bs = stats.get("chrf_delta_bootstrap", {})
    for cond_key, cond_label in [("without_tool", "Sin Herramienta"), ("with_tool", "Con Andamiaje (Ours)")]:
        s = stats[cond_key]
        summary_rows.append({
            "Modelo": model_name,
            "Condición": cond_label,
            "chrF++": s["mean_chrf"],
            "BLEU": s["mean_bleu"],
            "EPR (%)": s["epr_percent"],
            "CAR (%)": s["car_percent"] if s["car_percent"] is not None else "N/A",
            "Latencia (s)": s["mean_latency"],
            "Tokens/s": s["mean_tps"],
            "Δ chrF++ (Con - Sin)": f"{bs.get('delta', 0):+.2f} {bs.get('ci95', '')}" if cond_key == "with_tool" else "-",
            "p-valor": bs.get("p_value", "-") if cond_key == "with_tool" else "-"
        })

print("=== TABLA 1: RESUMEN DE MÉTRICAS PARA EL PAPER ===")
df_summary = pd.DataFrame(summary_rows)
display(df_summary)

# B) Tabla 2: Comparativa de Traducciones Oración por Oración
csv_path = "data/benchmark_results/translations_comparison_table.csv"
df_trans = pd.read_csv(csv_path)
print(f"\n=== TABLA 2: TRADUCCIONES LADO A LADO ({len(df_trans)} oraciones) ===")
display(df_trans)

=== TABLA 1: RESUMEN DE MÉTRICAS PARA EL PAPER ===


,Modelo,Condición,chrF++,BLEU,EPR (%),CAR (%),Latencia (s),Tokens/s,Δ chrF++ (Con - Sin),p-valor
0,llama3.2:3b,Sin Herramienta,11.62,2.10,100.0,N/A,0.18,224.69,-,-
1,llama3.2:3b,Con Andamiaje (Ours),16.59,3.80,100.0,57.1,1.54,215.72,"+4.97 [2.65, 7.24]",0.0
2,llama3.1:8b,Sin Herramienta,12.42,3.10,100.0,N/A,0.23,148.03,-,-
3,llama3.1:8b,Con Andamiaje (Ours),21.71,4.77,100.0,53.3,9.20,139.81,"+9.29 [6.77, 11.91]",0.0
4,qwen2.5:7b,Sin Herramienta,10.25,1.93,94.1,N/A,0.23,150.07,-,-
5,qwen2.5:7b,Con Andamiaje (Ours),18.16,3.08,88.2,53.3,9.09,139.91,"+7.90 [5.72, 10.28]",0.0
6,qwen2.5:14b,Sin Herramienta,11.52,2.40,94.1,N/A,0.39,83.88,-,-
7,qwen2.5:14b,Con Andamiaje (Ours),22.01,5.60,88.2,66.7,10.77,77.20,"+10.50 [7.88, 13.25]",0.0
8,aya-expanse:8b,Sin Herramienta,12.29,2.52,100.0,N/A,0.20,137.26,-,-
9,aya-expanse:8b,Con Andamiaje (Ours),19.83,3.64,82.4,64.3,1.66,129.27,"+7.54 [5.25, 9.83]",0.0



=== TABLA 2: TRADUCCIONES LADO A LADO (100 oraciones) ===


,id,quechua_source,spanish_reference,llama3.2_3b_with_tool,llama3.2_3b_without_tool,llama3.2_3b_chrf_with,llama3.2_3b_chrf_without,llama3.1_8b_with_tool,llama3.1_8b_without_tool,llama3.1_8b_chrf_with,llama3.1_8b_chrf_without,qwen2.5_7b_with_tool,qwen2.5_7b_without_tool,qwen2.5_7b_chrf_with,qwen2.5_7b_chrf_without,qwen2.5_14b_with_tool,qwen2.5_14b_without_tool,qwen2.5_14b_chrf_with,qwen2.5_14b_chrf_without,aya_expanse_8b_with_tool,aya_expanse_8b_without_tool,aya_expanse_8b_chrf_with,aya_expanse_8b_chrf_without,qwen2.5_32b_with_tool,qwen2.5_32b_without_tool,qwen2.5_32b_chrf_with,qwen2.5_32b_chrf_without,deepseek_r1_8b_with_tool,deepseek_r1_8b_without_tool,deepseek_r1_8b_chrf_with,deepseek_r1_8b_chrf_without
0,1,Noqayku wak’uysiykiku,Nosotros te ayudamos a cocinar,No hay nadie aquí,No hay que preocuparse por eso,8.04,9.79,No sabemos quién es,No tengo conocimientos en la lengua Quechua. ¿Puedo ayudarte con algo más?,10.67,16.84,Nosotros usamos cerámica antigua,"No sé cómo hacerlo porque no tengo conocimiento de esta frase en Quechua. Sin embargo, si me proporcionas más contexto o la frase completa, podré ayudarte mejor.",30.02,13.82,Estoy hablando de cerámica inca,Te quiero.,9.58,7.32,No nos creen,No nos equivocamos en nuestro camino.,6.13,16.46,Nosotros (sin ti) te ayudamos a cocinar.,Nosotros nos encontramos aquí.,82.46,31.13,Nosotros cocinamos su sopa,Los ayeres hablaron,37.03,9.46
1,2,Noqa ñakch’aykiwan ñakch’akuniraq,Yo aún me peino con tu peine,No hay nadie en la casa,"No hay que hacerlo todo a la vez"".",6.52,7.66,Nosotros nos hemos acostado,No podemos traducir contenido que promueva la violencia o el odio. ¿Hay algo más en lo que pueda ayudarte?,6.75,10.43,No sé quién es él o ella,No sé coser ni confecciono vestidos.,6.54,9.57,Ese hombre es el dueño de la casa,Yo no tengo hermanos.,5.48,10.91,"Nosotros, los que conocemos la verdad",No conocemos nuestras propias raíces,10.08,14.55,Yo me sigo peinando con tu peine.,Yo te veo a ti y tú me ves a mí.,54.07,11.89,Yo peino tu cabello y sigo peinándome,¡No te vayas disgustado/a!,26.75,5.66
2,3,Waqanayan,Quiere llorar,Tener deseos de llorar,El traductor experto en la lengua Quechua (variante Cusco-Collao y Sureño) te proporciona la traducción al español de la siguiente oración:,37.98,9.15,Lloramos,"La traducción de la oración ""Waqanayan"" al español es: ""Estoy cansado"".",22.13,8.75,Tienen deseos de llorar,Dijeron,37.40,6.62,Tengo ganas de llorar,Dios,36.61,3.85,Ellos desean llorar,Observan,34.66,6.50,Él/ella tiene deseos de llorar.,Mañana,33.20,1.32,Anhelar llorar,Nieve sagrada,34.92,7.48
3,4,Pay ña llank’asharanmi,El ya estaba trabajando,He visto una gran cantidad de personas,"Estoy cansado de trabajar"".",9.56,27.14,Yo estoy trabajando,No sé qué hacer.,40.80,4.03,Yo mismo vi la grada,"No sé cuál es el contexto exacto, pero traducido directamente podría ser: ""No sé cuánto tiempo"". Sin embargo, la traducción más natural podría ser: ""No sé cuánto tiempo ha pasado"".",6.55,9.03,Estoy usando la greda,Te deseo buena suerte,14.53,7.69,"Nosotros, con certeza, hemos visto la arcilla",No tengo palabras,7.92,8.63,Él/ella ya trabajó (lo sé con certeza),No me acuerdo,25.90,5.53,Él/ella ya está trabajando.,Usted nos toca,48.41,6.15
4,5,Waqanayashan,Está queriendo llorar,Tener deseos de llorar,El traductor experto en la lengua Quechua (variante Cusco-Collao y Sureño) te proporciona la traducción al español de la siguiente oración:,27.22,11.07,Están llorando,Están llorando,23.31,23.31,Tienen deseos de llorar,Dijeron,29.08,6.08,Tengo ganas de llorar,Dioses,24.77,5.28,Ellos desean llorar,Los guardianes,25.22,9.25,Él/ella está llorando o tiene deseos de llorar.,Nosotros creemos,31.17,6.18,Ella está queriendo llorar,Lugar sagrado del encuentro entre el mundo y la montaña,94.65,11.96
5,6,Noqanchis sarata mikhunchis,Nosotros comemos maíz,No hay dos maíces,No hay que asustarse,15.53,6.76,Nosotros hemos comido maíz con dos o más choclos,No hay nada que hacer.,43.88,4.69,Nosotros comemos el maíz,No

In [15]:
# 8. Ver las Tablas en formato LaTeX listas para el Paper
with open("docs/paper_tables.tex", "r") as f:
    latex_code = f.read()

print(latex_code)

% Tabla generada automáticamente para el Paper
\begin{table*}[t]
\centering
\small
\begin{tabular}{lcccccc}
\toprule
\textbf{Model} & \textbf{Condition} & \textbf{chrF++ $\uparrow$} & \textbf{BLEU $\uparrow$} & \textbf{EPR (\%) $\uparrow$} & \textbf{CAR (\%) $\uparrow$} & \textbf{Latency (s) $\downarrow$} \\
\midrule
\multirow{2}{*}{llama3.2 3b} & \textit{w/o Tool} & 11.62 & 2.10 & 100.0\% & -- & 0.18s \\
 & w/ Scaffold (Ours) & 16.59 & 3.80 & 100.0\% & 57.1\% & 1.54s \\
\midrule
\multirow{2}{*}{llama3.1 8b} & \textit{w/o Tool} & 12.42 & 3.10 & 100.0\% & -- & 0.23s \\
 & w/ Scaffold (Ours) & 21.71 & 4.77 & 100.0\% & 53.3\% & 9.20s \\
\midrule
\multirow{2}{*}{qwen2.5 7b} & \textit{w/o Tool} & 10.25 & 1.93 & 94.1\% & -- & 0.23s \\
 & w/ Scaffold (Ours) & 18.16 & 3.08 & 88.2\% & 53.3\% & 9.09s \\
\midrule
\multirow{2}{*}{qwen2.5 14b} & \textit{w/o Tool} & 11.52 & 2.40 & 94.1\% & -- & 0.39s \\
 & w/ Scaffold (Ours) & 22.01 & 5.60 & 88.2\% & 66.7\% & 10.77s \\
\midrule
\multirow{2}{*}{aya-e

In [16]:
# 9. Descargar artefactos generados como archivo ZIP
from google.colab import files
!zip -r quechua_academic_benchmark_results.zip data/benchmark_results/ docs/paper_tables.tex
files.download("quechua_academic_benchmark_results.zip")

  adding: data/benchmark_results/ (stored 0%)
  adding: data/benchmark_results/shard_3.json (deflated 80%)
  adding: data/benchmark_results/consolidated_benchmark_100.json (deflated 83%)
  adding: data/benchmark_results/translations_comparison_table.md (deflated 69%)
  adding: data/benchmark_results/shard_4.json (deflated 81%)
  adding: data/benchmark_results/translations_comparison_table.csv (deflated 65%)
  adding: data/benchmark_results/shard_1.json (deflated 80%)
  adding: data/benchmark_results/multi_model_benchmark.json (deflated 89%)
  adding: data/benchmark_results/shard_2.json (deflated 81%)
  adding: docs/paper_tables.tex (deflated 60%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>